Load and split by vehicle

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.append(str(Path.cwd().parent / "src"))
from features import FEATURES
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from xgboost import XGBClassifier

PROJ = Path.home() / "Documents/v2x-ids"
df = pd.read_parquet(PROJ / "data/processed/features_run79200.parquet")
X, y, groups = df[FEATURES], df["is_attack"], df["sender"]

# 80/20 split by SENDER; every attack type appears in both sets
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr, te = next(sgkf.split(X, df["label"], groups))
print("train", len(tr), "| test", len(te))
print("senders in both sets:", len(set(groups.iloc[tr]) & set(groups.iloc[te])))
print("classes in test:", df.iloc[te]["label"].nunique())

def report(name, yt, yp):
    print(f"{name:26s} P={precision_score(yt, yp):.3f}  R={recall_score(yt, yp):.3f}  F1={f1_score(yt, yp):.3f}")

train 125012 | test 32519
senders in both sets: 0
classes in test: 20


Rule-based baselines

In [2]:
def best_threshold(col):
    vals, yt = df.iloc[tr][col], y.iloc[tr]
    best_f1, best_t = 0, None
    for t in np.unique(np.quantile(vals.dropna(), np.linspace(0.5, 0.995, 100))):
        f = f1_score(yt, (vals > t).fillna(False).astype(int))
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

t_art, t_ssc = best_threshold("dist_to_receiver"), best_threshold("speed_mismatch")
print(f"ART threshold: {t_art:.1f} m | SSC threshold: {t_ssc:.2f} m/s")
test = df.iloc[te]
art = (test["dist_to_receiver"] > t_art).astype(int)
ssc = (test["speed_mismatch"] > t_ssc).fillna(False).astype(int)
report("Rule ART", y.iloc[te], art)
report("Rule SSC", y.iloc[te], ssc)
report("Rule ART OR SSC", y.iloc[te], art | ssc)

ART threshold: 149.9 m | SSC threshold: 1.15 m/s
Rule ART                   P=0.531  R=0.573  F1=0.551
Rule SSC                   P=0.967  R=0.614  F1=0.751
Rule ART OR SSC            P=0.624  R=0.862  F1=0.724


Random Forest and XGBoost

In [3]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced")
rf.fit(X.iloc[tr], y.iloc[tr])
p_rf = rf.predict(X.iloc[te])
report("Random Forest", y.iloc[te], p_rf)

ratio = (y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    tree_method="hist", scale_pos_weight=ratio, random_state=42)
xgb.fit(X.iloc[tr], y.iloc[tr])
report("XGBoost", y.iloc[te], xgb.predict(X.iloc[te]))

Random Forest              P=0.952  R=0.849  F1=0.898
XGBoost                    P=0.948  R=0.852  F1=0.898


The “cheating” random split, for comparison

In [4]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_rand = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42,
                                 class_weight="balanced").fit(Xtr, ytr)
report("RF, RANDOM split", yte, rf_rand.predict(Xte))

RF, RANDOM split           P=0.978  R=0.962  F1=0.970


Which attacks are caught, and false alarms

In [5]:
test = test.assign(pred=p_rf)
per_attack = test[test.is_attack == 1].groupby("attack_name")["pred"].mean().sort_values()
print((per_attack * 100).round(1))
print("False alarm rate:", round(test[test.is_attack == 0]["pred"].mean() * 100, 2), "%")
pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).head(8)

attack_name
ConstPosOffset          5.7
DataReplaySybil        39.6
DoSDisruptiveSybil     63.1
GridSybil              71.0
DataReplay             76.7
EventualStop           92.1
RandomPosOffset        95.6
ConstSpeed             96.2
Disruptive             97.3
DoS                    97.9
DoSDisruptive          98.2
DelayedMessages        98.4
ConstPos               99.4
RandomSpeedOffset      99.6
RandomPos              99.7
ConstSpeedOffset       99.8
DoSRandom              99.9
DoSRandomSybil         99.9
RandomSpeed           100.0
Name: pred, dtype: float64
False alarm rate: 3.22 %


speed_mismatch           0.236626
pos_pred_error           0.159023
implied_speed            0.092636
jump                     0.075054
dist_to_receiver         0.073292
heading_vs_motion_deg    0.073112
dt                       0.060443
speed                    0.051353
dtype: float64

In [6]:
results = pd.DataFrame({
    "model": ["Rule ART", "Rule SSC", "Rule ART OR SSC", "Random Forest", "XGBoost", "RF (random split)"],
    "f1": [f1_score(y.iloc[te], art), f1_score(y.iloc[te], ssc), f1_score(y.iloc[te], art | ssc),
           f1_score(y.iloc[te], p_rf), f1_score(y.iloc[te], xgb.predict(X.iloc[te])),
           f1_score(yte, rf_rand.predict(Xte))],
})
results.to_csv(PROJ / "results/task10_baselines.csv", index=False)
results.round(3)

,model,f1
0,Rule ART,0.551
1,Rule SSC,0.751
2,Rule ART OR SSC,0.724
3,Random Forest,0.898
4,XGBoost,0.898
5,RF (random split),0.970
